<a href="https://colab.research.google.com/github/VerginJenifer/policy-iteration-algorithm/blob/main/policy_iteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install git+https://github.com/mimoralea/gym-walk#egg=gym-walk

  Cloning https://github.com/mimoralea/gym-walk to /tmp/pip-install-5s2wp_id/gym-walk_e31a8755ad0d4920818edbd67292c8d2
  Running command git clone --filter=blob:none --quiet https://github.com/mimoralea/gym-walk /tmp/pip-install-5s2wp_id/gym-walk_e31a8755ad0d4920818edbd67292c8d2
  Resolved https://github.com/mimoralea/gym-walk to commit c453a5da6a37b6fadb393c490261bd6b9fa5bfc5
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.4/624.4 kB 12.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for gym-walk: filename=gym_walk-0.0.3-py3-none-any.whl size=5289 sha256=cfbe40f113c338b553a7a620b5bb4c16bf7e03e1c55bac19739b243d3dcdcc8f
  Stored in directory: /tmp/pip-ephem-wheel-cache-n9vvl_7e/wheels/bf/23/e5/a94be4a90dd18f7ce958c21f192276cb01ef0daaf

In [2]:
import gym
import numpy as np

import random
import warnings

warnings.filterwarnings('ignore', category=DeprecationWarning)
np.set_printoptions(suppress=True)
random.seed(123); np.random.seed(123);


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [81]:
def print_policy(pi, P, action_symbols=('<', 'v', '>', '^'), n_cols=4, title='Policy:'):
    print(title)
    arrs = {k:v for k,v in enumerate(action_symbols)}
    for s in range(len(P)):
        a = pi(s)
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), arrs[a].rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print("|")

In [56]:
def print_state_value_function(V, P, n_cols=4, prec=3, title='State-value function:'):
    print(title)
    for s in range(len(P)):
        v = V[s]
        print("| ", end="")
        if np.all([done for action in P[s].values() for _, _, _, done in action]):
            print("".rjust(9), end=" ")
        else:
            print(str(s).zfill(2), '{}'.format(np.round(v, prec)).rjust(6), end=" ")
        if (s + 1) % n_cols == 0: print(" | ")


In [57]:

def probability_success(env, pi, goal_state, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        while not done and steps < max_steps:
            state, _, done, h = env.step(pi(state))
            steps += 1
        results.append(state == goal_state)
    return np.sum(results)/len(results)

In [58]:
def mean_return(env, pi, n_episodes=100, max_steps=200):
    random.seed(123); np.random.seed(123) ; env.seed(123)
    results = []
    for _ in range(n_episodes):
        state, done, steps = env.reset(), False, 0
        results.append(0.0)
        while not done and steps < max_steps:
            state, reward, done, _ = env.step(pi(state))
            results[-1] += reward
            steps += 1
    return np.mean(results)

In [59]:
env = gym.make('FrozenLake-v1')
P = env.env.P
init_state = env.reset()
goal_state = 15
LEFT, DOWN, RIGHT, UP = range(4)

In [60]:
P

{0: {0: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  2: [(0.3333333333333333, 4, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)],
  3: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 1: {0: [(0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True)],
  1: [(0.3333333333333333, 0, 0.0, False),
   (0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False)],
  2: [(0.3333333333333333, 5, 0.0, True),
   (0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False)],
  3: [(0.3333333333333333, 2, 0.0, False),
   (0.3333333333333333, 1, 0.0, False),
   (0.3333333333333333, 0, 0.0, False)]},
 2:

In [61]:
state, reward, done, info = env.step(RIGHT)
print("state:{0} - reward:{1} - done:{2} - info:{3}".format(state, reward, done, info))

state:1 - reward:0.0 - done:False - info:{'prob': 0.3333333333333333}


In [62]:
pi_frozenlake1 = lambda s: {
    0: RIGHT,
    1: RIGHT,
    2: RIGHT,
    3: RIGHT,
    4: RIGHT,
    5: RIGHT,
    6: RIGHT,
    7: RIGHT,
    8: RIGHT,
    9: RIGHT,
    10:RIGHT,
    11:RIGHT,
    12:RIGHT,
    13:RIGHT,
    14:RIGHT,
    15:RIGHT #Stop
}[s]
print("Name: D Vergin Jenifer     ")
print("Register Number: 212223240174         ")
print_policy(pi_frozenlake1, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)

Name: D Vergin Jenifer     
Register Number: 212223240174         
Policy:
| 00      > | 01      > | 02      > | 03      > |
| 04      > |           | 06      > |           |
| 08      > | 09      > | 10      > |           |
|           | 13      > | 14      > |           |


In [63]:
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, pi_frozenlake1, goal_state=goal_state)*100,
    mean_return(env, pi_frozenlake1)))

Reaches goal 1.00%. Obtains an average undiscounted return of 0.0100.


In [65]:
def policy_evaluation(pi, P, gamma=1.0, theta=1e-10):
    prev_V = np.zeros(len(P), dtype=np.float64)
    while True:
        V = np.zeros(len(P), dtype=np.float64)
        for s in range(len(P)):
            for prob, next_state, reward, done in P[s][pi(s)]:
                V[s] += prob * (
                    reward + gamma * prev_V[next_state] * (not done)
                )
        if np.max(np.abs(prev_V - V)) < theta:
            break
        prev_V = V.copy()
    return V

In [66]:
V1 = policy_evaluation(pi_frozenlake1, P)
print("Name: D Vergin Jenifer    ")
print("Register Number: 212223240174        ")
print_state_value_function(V1, P, n_cols=4, prec=5)

Name: D Vergin Jenifer    
Register Number: 212223240174        
State-value function:
| 00 0.0315 | 01 0.02381 | 02 0.04762 | 03    0.0  | 
| 04 0.03919 |           | 06 0.09524 |            | 
| 08 0.08608 | 09 0.21905 | 10 0.2381 |            | 
|           | 13 0.41905 | 14 0.61905 |            | 


In [70]:
def policy_improvement(V, P, gamma=1.0):
    Q = np.zeros((len(P), len(P[0])), dtype=np.float64)
    for s in range(len(P)):
        for a in range(len(P[s])):
            for prob, next_state, reward, done in P[s][a]:
                Q[s][a] += prob * (reward + gamma * V[next_state] * (not done))
    new_pi = lambda s: np.argmax(Q[s])
    return new_pi

In [71]:
pi_2 = policy_improvement(V1, P)
print("Name: D Vergin Jenifer     ")
print("Register Number: 212223240174        ")
print_policy(pi_2, P, action_symbols=('<', 'v', '>', '^'), n_cols=4)

Name: D Vergin Jenifer     
Register Number: 212223240174        
Policy:
| 00      < | 01      ^ | 02      < | 03      < |
| 04      < |           | 06      < |           |
| 08      ^ | 09      v | 10      < |           |
|           | 13      > | 14      v |           |


In [72]:
print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, pi_2, goal_state=goal_state)*100,
    mean_return(env, pi_2)))

Reaches goal 66.00%. Obtains an average undiscounted return of 0.6600.


In [73]:

V2 = policy_evaluation(pi_2, P)
print("Name: D Vergin Jenifer     ")
print("Register Number: 212223240174         ")
print_state_value_function(V2, P, n_cols=4, prec=5)


Name: D Vergin Jenifer     
Register Number: 212223240174         
State-value function:
| 00 0.78049 | 01 0.65854 | 02 0.53659 | 03 0.26829  | 
| 04 0.78049 |           | 06 0.41463 |            | 
| 08 0.78049 | 09 0.78049 | 10 0.70732 |            | 
|           | 13 0.85366 | 14 0.92683 |            | 


In [74]:
def policy_iteration(P, gamma=1.0, theta=1e-10):
  random_actions = np.random.choice(tuple(P[0].keys()), len(P))
  pi=lambda s: {s: a for s, a in enumerate(random_actions)}[s]
  while True:
    old_pi= {s:pi(s) for s in range(len(P))}
    V=policy_evaluation(pi,P,gamma,theta)
    pi=policy_improvement(V,P,gamma)
    if old_pi == {s: pi(s) for s in range(len(P))}:
      break
  return V, pi

In [75]:
optimal_V, optimal_pi = policy_iteration(P)

In [84]:

print("Name: D Vergin Jenifer     ")
print("Register Number: 212223240174         ")
print('Optimal policy and state-value function (PI):')
print_policy(optimal_pi, P,action_symbols=('<', 'v', '>', '^'),n_cols=4)


Name: D Vergin Jenifer     
Register Number: 212223240174         
Optimal policy and state-value function (PI):
Policy:
| 00      < | 01      ^ | 02      ^ | 03      ^ |
| 04      < |           | 06      < |           |
| 08      ^ | 09      v | 10      < |           |
|           | 13      > | 14      v |           |


In [85]:

print('Reaches goal {:.2f}%. Obtains an average undiscounted return of {:.4f}.'.format(
    probability_success(env, optimal_pi, goal_state=goal_state)*100,
    mean_return(env, optimal_pi)))

Reaches goal 69.00%. Obtains an average undiscounted return of 0.6900.


In [86]:
print("Name: D Vergin Jenifer     ")
print("Register Number: 212223240174         ")
print_state_value_function(optimal_V, P, n_cols=4, prec=5)

Name: D Vergin Jenifer     
Register Number: 212223240174         
State-value function:
| 00 0.82353 | 01 0.82353 | 02 0.82353 | 03 0.82353  | 
| 04 0.82353 |           | 06 0.52941 |            | 
| 08 0.82353 | 09 0.82353 | 10 0.76471 |            | 
|           | 13 0.88235 | 14 0.94118 |            | 
